In [1]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 10.9 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.4 MB 31.4 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 28.2 MB/s  0:00:00
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   ------------------------ -----------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install category_encoders

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from category_encoders.count import CountEncoder

In [5]:
DATA_PATH = Path("../data/raw/songs/Music Info.csv")

songs = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {songs.shape}")
songs.head()

Dataset shape: (50683, 21)


,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...,09ZQ5TmUG8TSL56n0knqrj,"rock, alternative, indie, alternative_rock, in...",NaN,2004,222200,0.355,...,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...,06UfBBDISthj1ZJAtX4xjj,"rock, alternative, indie, pop, alternative_roc...",NaN,2006,258613,0.409,...,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,0keNu0t0tqsWtExGM3nT1D,"rock, alternative, alternative_rock, 90s, grunge",RnB,1991,218920,0.508,...,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...,0ancVQ9wEcHVd0RrGICTE4,"rock, alternative, indie, alternative_rock, in...",NaN,2004,237026,0.279,...,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,01QoK9DA7VTeTSE3MNzp4I,"rock, alternative, indie, alternative_rock, in...",RnB,2008,238640,0.515,...,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4


In [6]:
songs.drop_duplicates(
    subset=["spotify_id", "year", "duration_ms"],
    inplace=True
)

songs.reset_index(drop=True, inplace=True)

print(f"Dataset shape after removing duplicates: {songs.shape}")

Dataset shape after removing duplicates: (50674, 21)


## Feature Selection

Only attributes that describe the characteristics of a song are used to calculate similarity. Identifiers, song titles, preview URLs, and the sparse `genre` feature are excluded from the feature matrix.

In [7]:
columns_to_remove = [
    "track_id",
    "name",
    "spotify_preview_url",
    "spotify_id",
    "genre"
]

content_features = songs.drop(columns=columns_to_remove).copy()

content_features.head()

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,Nirvana,"rock, alternative, alternative_rock, 90s, grunge",1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,Franz Ferdinand,"rock, alternative, indie, alternative_rock, in...",2004,237026,0.279,0.664,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,Radiohead,"rock, alternative, indie, alternative_rock, in...",2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4


In [8]:
content_features.isna().sum()

artist                 0
tags                1126
year                   0
duration_ms            0
danceability           0
energy                 0
key                    0
loudness               0
mode                   0
speechiness            0
acousticness           0
instrumentalness       0
liveness               0
valence                0
tempo                  0
time_signature         0
dtype: int64

In [9]:
content_features["tags"] = content_features["tags"].fillna("no_tags")

print(f"Missing values remaining: {content_features.isna().sum().sum()}")

Missing values remaining: 0


In [10]:
content_features["artist"] = content_features["artist"].str.lower()
content_features["tags"] = content_features["tags"].str.lower()

## Feature Transformation

Different feature types require different preprocessing:

- `year` is frequency encoded.
- `artist`, `key`, and `time_signature` are one-hot encoded.
- `tags` are converted into TF-IDF features.
- `duration_ms`, `loudness`, and `tempo` are standardized.
- Audio features bounded approximately between 0 and 1 are min-max scaled.
- `mode` is already binary and is passed through unchanged.

In [11]:
frequency_encode_columns = ["year"]

one_hot_columns = [
    "artist",
    "time_signature",
    "key"
]

tfidf_column = "tags"

standard_scale_columns = [
    "duration_ms",
    "loudness",
    "tempo"
]

min_max_scale_columns = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence"
]

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "year_frequency",
            CountEncoder(normalize=True, return_df=True),
            frequency_encode_columns
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            one_hot_columns
        ),
        (
            "tags_tfidf",
            TfidfVectorizer(max_features=85),
            tfidf_column
        ),
        (
            "standard_scaling",
            StandardScaler(),
            standard_scale_columns
        ),
        (
            "minmax_scaling",
            MinMaxScaler(),
            min_max_scale_columns
        )
    ],
    remainder="passthrough",
    n_jobs=-1
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('year_frequency', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transforme

In [13]:
transformed_features = preprocessor.fit_transform(content_features)

print(f"Original shape: {content_features.shape}")
print(f"Transformed shape: {transformed_features.shape}")

Original shape: (50674, 16)
Transformed shape: (50674, 8431)


In [14]:
songs.loc[
    songs["name"].str.lower() == "whenever, wherever",
    ["name", "artist", "year", "tags"]
]

,name,artist,year,tags
1025,"Whenever, Wherever",Shakira,2012,"rock, pop, female_vocalists, singer_songwriter..."


In [15]:
song_index = songs.index[
    songs["name"].str.lower() == "whenever, wherever"
][0]

song_index

1025

In [16]:
song_vector = transformed_features[song_index]

similarity_scores = cosine_similarity(
    song_vector,
    transformed_features
).ravel()

similarity_scores.shape

(50674,)

In [17]:
ranked_indices = np.argsort(similarity_scores)[::-1]

recommended_indices = ranked_indices[
    ranked_indices != song_index
][:10]

songs.loc[
    recommended_indices,
    ["name", "artist", "tags"]
]

,name,artist,tags
12305,Why Wait,Shakira,"pop, experimental, singer_songwriter, dance"
6046,Hips Don't Lie,Shakira,"pop, female_vocalists, singer_songwriter, danc..."
6129,Oops!...I Did It Again,Britney Spears,"pop, female_vocalists, dance, 00s"
17241,Perfect Lover,Britney Spears,"pop, dance, rnb, 00s"
6133,Bootylicious,Destiny's Child,"pop, female_vocalists, dance, soul, hip_hop, r..."
7172,Wild Things,Alessia Cara,"pop, female_vocalists"
6121,La Isla Bonita,Madonna,"pop, female_vocalists, dance, 80s"
6526,Cruel Summer,Bananarama,"pop, female_vocalists, dance, 80s, new_wave"
38383,Dreams for Plans,Shakira,"pop, female_vocalists, guitar, pop_rock"
6287,Trick Me,Kelis,"pop, female_vocalists, dance, soul, hip_hop, rnb"


In [18]:
def recommend_songs(song_name, songs_data, feature_matrix, k=10):
    song_matches = songs_data[
        songs_data["name"].str.lower() == song_name.lower()
    ]

    if song_matches.empty:
        print(f"'{song_name}' was not found in the catalogue.")
        return None

    song_index = song_matches.index[0]

    similarity_scores = cosine_similarity(
        feature_matrix[song_index],
        feature_matrix
    ).ravel()

    ranked_indices = np.argsort(similarity_scores)[::-1]

    recommended_indices = ranked_indices[
        ranked_indices != song_index
    ][:k]

    recommendations = songs_data.loc[
        recommended_indices,
        ["name", "artist", "spotify_preview_url"]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[
        recommended_indices
    ]

    return recommendations.reset_index(drop=True)

In [19]:
recommend_songs(
    song_name="Whenever, Wherever",
    songs_data=songs,
    feature_matrix=transformed_features,
    k=10
)

,name,artist,spotify_preview_url,similarity_score
0,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...,1.0
1,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,1.0
2,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...,1.0
3,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...,1.0
4,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...,1.0
5,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...,1.0
6,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...,1.0
7,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...,1.0
8,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...,1.0
9,Trick Me,Kelis,https://p.scdn.co/mp3-preview/0862413f9e28d284...,1.0


In [20]:
pd.Series(
    similarity_scores[recommended_indices]
).map(lambda x: f"{x:.10f}")

0    0.9999997885
1    0.9999997160
2    0.9999996909
3    0.9999996378
4    0.9999996191
5    0.9999996165
6    0.9999996142
7    0.9999996139
8    0.9999996041
9    0.9999995954
dtype: str

In [21]:
print("Maximum similarity:", similarity_scores[recommended_indices].max())
print("Minimum similarity:", similarity_scores[recommended_indices].min())
print("Unique similarity values:", np.unique(similarity_scores[recommended_indices]))

Maximum similarity: 0.9999997885111389
Minimum similarity: 0.999999595366365
Unique similarity values: [0.9999996  0.9999996  0.99999961 0.99999961 0.99999962 0.99999962
 0.99999964 0.99999969 0.99999972 0.99999979]
